# 02 Inventory Scan

This notebook performs an **inventory-only** scan. It does **not** rename, move, or delete anything.

Use it on a **small sandbox copy first**, then review the CSV/Parquet outputs inside VS Code before moving to classification.

## Safety checks

- Scan a copied sandbox folder first.
- Keep `WRITE_OUTPUTS = True` so results are persisted for review.
- Do not point the first run at your live master tree.

In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'src').exists(), f'Could not locate project root from {Path.cwd()}'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.inventory import InventoryScanner

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 120)

print(f'Project root: {PROJECT_ROOT}')

Project root: c:\00_Developement\sch-file-organizer


In [2]:
# Optional demo sandbox creator for a smoke test.
# Run this cell only if you want a tiny synthetic folder tree to test the notebook end-to-end.

demo_root = PROJECT_ROOT / 'data' / 'working' / 'demo_sandbox'
demo_root.mkdir(parents=True, exist_ok=True)

samples = {
    demo_root / 'misc' / 'readme.txt': 'Demo text file for inventory scan.\n',
    demo_root / 'downloads' / 'Thumbs.db': b'',
    demo_root / 'drawings' / 'DTC12p500-01_FS_DRWTEC_SITE-LAYOUT_20260307_v01_WIP.pdf': b'%PDF-1.4 demo\n',
    demo_root / 'duplicates' / 'copy-a.txt': 'same content\n',
    demo_root / 'duplicates' / 'copy-b.txt': 'same content\n',
}

for path, content in samples.items():
    path.parent.mkdir(parents=True, exist_ok=True)
    if isinstance(content, bytes):
        path.write_bytes(content)
    else:
        path.write_text(content, encoding='utf-8')

print(f'Demo sandbox ready: {demo_root}')

Demo sandbox ready: c:\00_Developement\sch-file-organizer\data\working\demo_sandbox


In [3]:
# Configure the scan target here.
# Start with the demo sandbox or a small copied sandbox from your real files.

SCAN_ROOT = PROJECT_ROOT / 'data' / 'working' / 'demo_sandbox'
# Example for a real sandbox copy on Windows:
# SCAN_ROOT = Path(r'D:\\SCH_SANDBOX_SAMPLE')

WRITE_OUTPUTS = True
INCLUDE_HIDDEN = False
HASH_FILES = True
FOLLOW_SYMLINKS = False

SCAN_ROOT = SCAN_ROOT.expanduser().resolve()
assert SCAN_ROOT.exists() and SCAN_ROOT.is_dir(), f'Scan root not found or not a directory: {SCAN_ROOT}'
print(f'Scan root: {SCAN_ROOT}')

Scan root: C:\00_Developement\sch-file-organizer\data\working\demo_sandbox


In [4]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_dir = PROJECT_ROOT / 'data' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / f'inventory_{timestamp}.csv'
parquet_path = output_dir / f'inventory_{timestamp}.parquet'

scanner = InventoryScanner(
    root_path=SCAN_ROOT,
    include_hidden=INCLUDE_HIDDEN,
    hash_files=HASH_FILES,
    follow_symlinks=FOLLOW_SYMLINKS,
)

inventory_df = scanner.write_outputs(
    csv_path=csv_path if WRITE_OUTPUTS else None,
    parquet_path=parquet_path if WRITE_OUTPUTS else None,
)

print(f'Rows: {len(inventory_df):,}')
print(f'CSV: {csv_path}')
print(f'Parquet: {parquet_path}')
inventory_df.head(10)

Rows: 5
CSV: c:\00_Developement\sch-file-organizer\data\outputs\inventory_20260307_074202.csv
Parquet: c:\00_Developement\sch-file-organizer\data\outputs\inventory_20260307_074202.parquet


,absolute_path,root_path,relative_path,parent_path,name,stem,extension,size_bytes,modified_utc,created_utc,depth,content_hash,is_hidden,is_symlink
0,C:\00_Developement\sch-file-organizer\data\working\demo_sandbox\downloads\Thumbs.db,C:\00_Developement\sch-file-organizer\data\working\demo_sandbox,downloads/Thumbs.db,downloads,Thumbs.db,Thumbs,.db,0,2026-03-07T05:41:33.767030+00:00,2026-03-07T03:38:10+00:00,2,cae66941d9efbd404e4d88758ea67670,False,False
1,C:\00_Developement\sch-file-organizer\data\working\demo_sandbox\drawings\DTC12p500-01_FS_DRWTEC_SITE-LAYOUT_20260307...,C:\00_Developement\sch-file-organizer\data\working\demo_sandbox,drawings/DTC12p500-01_FS_DRWTEC_SITE-LAYOUT_20260307_v01_WIP.pdf,drawings,DTC12p500-01_FS_DRWTEC_SITE-LAYOUT_20260307_v01_WIP.pdf,DTC12p500-01_FS_DRWTEC_SITE-LAYOUT_20260307_v01_WIP,.pdf,14,2026-03-07T05:41:33.767030+00:00,2026-03-07T03:38:10+00:00,2,9ff079f7c2e7c32ce5a0e4841bcd78c8,False,False
2,C:\00_Developement\sch-file-organizer\data\working\demo_sandbox\duplicates\copy-a.txt,C:\00_Developement\sch-file-organizer\data\working\demo_sandbox,duplicates/copy-a.txt,duplicates,copy-a.txt,copy-a,.txt,14,2026-03-07T05:41:33.768032+00:00,2026-03-07T03:38:10+00:00,2,bd4acf07f10ff0c66c7a599c5e70760e,False,False
3,C:\00_Developement\sch-file-organizer\data\working\demo_sandbox\duplicates\copy-b.txt,C:\00_Developement\sch-file-organizer\data\working\demo_sandbox,duplicates/copy-b.txt,duplicates,copy-b.txt,copy-b,.txt,14,2026-03-07T05:41:33.769031+00:00,2026-03-07T03:38:10+00:00,2,bd4acf07f10ff0c66c7a599c5e70760e,False,False
4,C:\00_Developement\sch-file-organizer\data\working\demo_sandbox\misc\readme.txt,C:\00_Developement\sch-file-organizer\data\working\demo_sandbox,misc/readme.txt,misc,readme.txt,readme,.txt,36,2026-03-07T05:41:33.766033+00:00,2026-03-07T03:38:10+00:00,2,08fdf54ca2826d5d068cee33a8e04a75,False,False


## Quick quality checks

These checks help verify that the scan looks sane before moving on to classification.

In [5]:
summary = {
    'file_count': int(len(inventory_df)),
    'total_size_mb': round(float(inventory_df['size_bytes'].sum()) / (1024 * 1024), 2),
    'hidden_files': int(inventory_df['is_hidden'].sum()),
    'symlinks': int(inventory_df['is_symlink'].sum()),
    'max_depth': int(inventory_df['depth'].max()) if len(inventory_df) else 0,
    'missing_hashes': int(inventory_df['content_hash'].isna().sum()) if 'content_hash' in inventory_df else 0,
}
pd.Series(summary, name='inventory_summary')

file_count        5.0
total_size_mb     0.0
hidden_files      0.0
symlinks          0.0
max_depth         2.0
missing_hashes    0.0
Name: inventory_summary, dtype: float64

In [6]:
inventory_df['extension'] = inventory_df['extension'].fillna('')
extension_summary = (
    inventory_df.groupby('extension', dropna=False)
    .agg(file_count=('name', 'count'), total_size_mb=('size_bytes', lambda s: round(s.sum() / (1024 * 1024), 2)))
    .sort_values(['file_count', 'total_size_mb'], ascending=[False, False])
    .reset_index()
)
extension_summary.head(20)

,extension,file_count,total_size_mb
0,.txt,3,0.0
1,.db,1,0.0
2,.pdf,1,0.0


In [7]:
duplicate_df = (
    inventory_df.dropna(subset=['content_hash'])
    .groupby('content_hash')
    .agg(
        file_count=('absolute_path', 'count'),
        total_size_mb=('size_bytes', lambda s: round(s.sum() / (1024 * 1024), 2)),
        sample_paths=('relative_path', lambda s: list(s.head(5))),
    )
    .reset_index()
    .query('file_count > 1')
    .sort_values(['file_count', 'total_size_mb'], ascending=[False, False])
)

duplicate_df.head(20)

,content_hash,file_count,total_size_mb,sample_paths
2,bd4acf07f10ff0c66c7a599c5e70760e,2,0.0,"[duplicates/copy-a.txt, duplicates/copy-b.txt]"


In [12]:
path_risk_df = inventory_df.copy()
path_risk_df['absolute_path_len'] = path_risk_df['absolute_path'].str.len()
(
    path_risk_df[['absolute_path_len', 'size_bytes', 'relative_path', 'name']]
    .sort_values('absolute_path_len', ascending=False)
    .head(20)
)

,absolute_path_len,size_bytes,relative_path,name
1,128,14,drawings/DTC12p500-01_FS_DRWTEC_SITE-LAYOUT_20260307_v01_WIP.pdf,DTC12p500-01_FS_DRWTEC_SITE-LAYOUT_20260307_v01_WIP.pdf
3,85,14,duplicates/copy-b.txt,copy-b.txt
2,85,14,duplicates/copy-a.txt,copy-a.txt
0,83,0,downloads/Thumbs.db,Thumbs.db
4,79,36,misc/readme.txt,readme.txt


In [11]:
(
    inventory_df[['relative_path', 'name', 'extension', 'size_bytes', 'modified_utc', 'content_hash']]
    .sort_values(['extension', 'name'])
    .head(30))

,relative_path,name,extension,size_bytes,modified_utc,content_hash
0,downloads/Thumbs.db,Thumbs.db,.db,0,2026-03-07T05:41:33.767030+00:00,cae66941d9efbd404e4d88758ea67670
1,drawings/DTC12p500-01_FS_DRWTEC_SITE-LAYOUT_20260307_v01_WIP.pdf,DTC12p500-01_FS_DRWTEC_SITE-LAYOUT_20260307_v01_WIP.pdf,.pdf,14,2026-03-07T05:41:33.767030+00:00,9ff079f7c2e7c32ce5a0e4841bcd78c8
2,duplicates/copy-a.txt,copy-a.txt,.txt,14,2026-03-07T05:41:33.768032+00:00,bd4acf07f10ff0c66c7a599c5e70760e
3,duplicates/copy-b.txt,copy-b.txt,.txt,14,2026-03-07T05:41:33.769031+00:00,bd4acf07f10ff0c66c7a599c5e70760e
4,misc/readme.txt,readme.txt,.txt,36,2026-03-07T05:41:33.766033+00:00,08fdf54ca2826d5d068cee33a8e04a75


## Exit criteria for this notebook

Move to the next phase only when all of these are true:

1. The scan root is the folder you intended.
2. File counts and extensions look plausible.
3. Duplicate detection behaves as expected.
4. No surprising permission or path issues appear.
5. CSV and Parquet outputs were written successfully.

Once this passes on a sandbox copy, the next step is a deterministic classification notebook driven by `rules.py`.